In [ ]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from IPython.core.interactiveshell import re
import os
# make sure path /content/drive/MyDrive/dit_fine_resluts/ exists
res_dir = '/content/drive/MyDrive/dit_fine_resluts/'
os.makedirs(res_dir, exist_ok=True)

In [ ]:
# Install dependencies
!pip install timm diffusers

In [ ]:
# IMPORTANT: This notebook is designed for YOUR CUSTOM DiT fork with x2 modifications.
#
# Option 1: If you've pushed your code to GitHub, replace the URL below:
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git
# %cd YOUR_REPO
#
# Option 2: For now, we clone the base DiT repo and overwrite with your custom files:
!git clone -b vit-side https://github.com/mrmc-speco/DiT.git
%cd DiT
!git checkout vit-side
!git pull

# The next cells will overwrite models.py (with x2 modifications) and add new scripts

Cloning into 'DiT'...
remote: Enumerating objects: 432, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 432 (delta 0), reused 0 (delta 0), pack-reused 431 (from 2)
Receiving objects: 100% (432/432), 30.90 MiB | 9.29 MiB/s, done.
Resolving deltas: 100% (247/247), done.
/content/DiT
Already on 'vit-side'
Your branch is up to date with 'origin/vit-side'.
Already up to date.


In [ ]:
%cd /content/DiT
!pwd
!git pull

/content/DiT
/content/DiT
Already up to date.


In [ ]:
# Verify that the freezing logic works correctly
!python test_freezing.py

Initializing model...
Creating DiT-XL/2 model...
[DiT] Loading pre-trained ViT model: vit_large_patch16_224
model.safetensors: 100% 1.22G/1.22G [00:03<00:00, 333MB/s]
[DiT] Pre-trained ViT block: dim=1024, num_heads=16
[DiT] Target x2_vit_block: dim=1152, num_heads=16
[DiT] Dimensions don't match. Creating block with pretrained dimensions and projection layers...
[DiT] ✓ Successfully loaded pre-trained ViT weights into block with dim=1024
[DiT] Using projection layers to adapt dimensions: 1152 -> 1024 -> 1152
Applying freezing logic...
Verifying parameters...
SUCCESS: Freezing logic verified correctly.
Total Params: 690,093,088
Trainable Params: 17,656,864 (2.56%)


In [ ]:
# Download ImageNet Validation Dataset (ILSVRC2012)
# This will download ~6.3GB and organize it into class folders

import os
import tarfile
from pathlib import Path
import xml.etree.ElementTree as ET

# Download validation set
print('Downloading ImageNet validation set (~6.3GB)...')
!wget -nc https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_val.tar

# Download validation ground truth annotations
print('Downloading validation ground truth...')
!wget -nc https://image-net.org/data/ILSVRC/2012/ILSVRC2012_devkit_t12.tar.gz

# Extract validation images
print('Extracting validation images...')
os.makedirs('imagenet_val', exist_ok=True)
!tar -xf ILSVRC2012_img_val.tar -C imagenet_val

# Extract devkit to get ground truth
print('Extracting devkit...')
!tar -xzf ILSVRC2012_devkit_t12.tar.gz

# Parse ground truth from devkit
print('Parsing validation ground truth...')
gt_file = 'ILSVRC2012_devkit_t12/data/ILSVRC2012_validation_ground_truth.txt'
with open(gt_file, 'r') as f:
    # Ground truth file has 1-indexed class IDs, convert to 0-indexed
    labels = [int(line.strip()) - 1 for line in f]

# Organize into class folders
print('Organizing into class folders...')
val_dir = Path('imagenet_val')
organized_dir = Path('imagenet_val_organized')

# Create class directories
for class_id in set(labels):
    (organized_dir / str(class_id)).mkdir(parents=True, exist_ok=True)

# Move images to class folders
val_images = sorted(val_dir.glob('ILSVRC2012_val_*.JPEG'))
for idx, img_path in enumerate(val_images):
    class_id = labels[idx]
    target_path = organized_dir / str(class_id) / img_path.name
    img_path.rename(target_path)

print(f'✓ ImageNet validation set organized into {len(set(labels))} class folders')
print(f'Total images: {len(val_images)}')
print('Dataset ready at: ./imagenet_val_organized')

--2025-12-09 18:00:41--  https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_val.tar
Resolving image-net.org (image-net.org)... 171.64.68.16
Connecting to image-net.org (image-net.org)|171.64.68.16|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6744924160 (6.3G) [application/x-tar]
Saving to: ‘ILSVRC2012_img_val.tar’

ILSVRC2012_img_val. 100%[===================>]   6.28G  4.42MB/s    in 21m 4s  

2025-12-09 18:21:46 (5.09 MB/s) - ‘ILSVRC2012_img_val.tar’ saved [6744924160/6744924160]

--2025-12-09 18:21:46--  https://image-net.org/data/ILSVRC/2012/ILSVRC2012_devkit_t12.tar.gz
Resolving image-net.org (image-net.org)... 171.64.68.16
Connecting to image-net.org (image-net.org)|171.64.68.16|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2568145 (2.4M) [application/x-gzip]
Saving to: ‘ILSVRC2012_devkit_t12.tar.gz’

ILSVRC2012_devkit_t 100%[===================>]   2.45M  2.07MB/s    in 1.2s    

2025-12-09 18:21:48 (2.07 MB/s) - ‘ILSVR

In [ ]:
# Custom results directory
!torchrun --nnodes=1 --nproc_per_node=1 train_x2_finetune.py \
    --model DiT-XL/2 \
    --pretrained-ckpt DiT-XL-2-256x256.pt \
    --data-path ./imagenet_val_organized \
    --results-dir /content/drive/MyDrive/dit_fine_resluts/ \
    --classes 972 973 974 975 976 \
    --epochs 1400 \
    --global-batch-size 50 \
    --log-every 5 \
    --ckpt-every 500

/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)
2025-12-09 18:22:25.072828: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-09 18:22:25.089902: E external/local_xla/xla/stream_executor/cuda/cuda_f